# Классификация проектов с помощью линейной регрессии

In [1]:
# install libraries

%pip install numpy==1.23.5
%pip install typer==0.9.4
%pip install torch==2.0.1
%pip install transformers==4.34.0
%pip install sentence-transformers==3.0.0
%pip install spacy==3.5.4
%pip install tensorflow==2.12.0
%pip install torchtext==0.15.2

%pip check

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --up

In [2]:
# library deps
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import nltk
import pandas as pd
from gensim.models.word2vec import Word2Vec
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from xgboost import XGBClassifier

2026-05-09 10:47:39.130327: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [20]:
import xgboost as xgb
print(xgb.__version__)

1.7.6


## Загрузка данных

In [3]:
Labels = [
    "Автомобильные дороги",
    "Водоотведение",
    "Водопроводы",
    "Газоны дорожки",
    "Газопроводы",
    "Горные выработки",
    "Железнодорожные пути",
    "Заводы фабрики",
    "Здания",
    "Инженерное обеспечение",
    "Инфраструктура наземного электротранспорта",
    "Линии электропередачи",
    "Метрополитены",
    "Мосты и тоннели",
    "Наружное освещение",
    "Нефтепроводы",
    "Сооружения",
    "Теплопроводы",
    "Технологические установки",
]

ColumnNames = ["id", "project_name", "label"]


def load_labeled_data(path):
    labeled_dataframes = [
        pd.read_csv(f"{path}//{label}.csv", names=ColumnNames, header=0)
        for label in tqdm(Labels)
    ]
    result_df = pd.concat(labeled_dataframes)
    result_df["project_name"] = result_df["project_name"].str.strip('"')
    return result_df


def load_unlabeled_data(path):
    return pd.read_csv(
        path, sep=";", encoding="utf-8", nrows=200000, names=["id", "project_name"]
    )


# raw_df = load_unlabeled_data(f'../Data/Реестр 2022-2024 clean.csv')
# raw_df

## Разделение данных на тестовую и обучающую выборки

In [4]:
def data_train_test_split(data, labels):
    assert len(data) == len(
        labels
    ), "Размеры списков данных и результатов разметки не совпадают"
    le = LabelEncoder()
    le.fit(labels)
    y = le.transform(labels)
    return train_test_split(data, y, test_size=0.2, random_state=42)

## Способы векторизации

In [5]:
def vectorize_words_with_word2vec(sentences, vector_size):
    nltk.download("punkt")
    tokenized_sentences = [
        nltk.tokenize.word_tokenize(text.lower(), language="russian")
        for text in tqdm(sentences)
    ]
    sentence_vectors = Word2Vec(
        tokenized_sentences,
        workers=8,
        vector_size=vector_size,
        min_count=3,
        window=5,
        epochs=15,
    )
    return sentence_vectors


def vectorize_with_word2vec(sentences, vector_size):
    result = []
    for word in word_tokenize(text.lower()):
        if word in model_tweets.wv:
            result.append(model_tweets.wv[word])

    if len(result):
        result = np.average(result, axis=0)
    else:
        result = np.zeros(300)
    return result

In [6]:
# Вернет матрицу размера (len(sentences, 1024)
def vectorize_with_sentence_transformer(model_name, sentences):
    model = SentenceTransformer(model_name)
    return model.encode(sentences.to_numpy())

## Модели

In [21]:
# Логистическая регрессия
def log_reg(X_train, y_train):
    model = LogisticRegression()
    model.fit(X_train, y_train)
    return model


# XGBoost
def xgb(X_train, y_train):
    xgb = XGBClassifier(use_label_encoder=False, n_jobs=-1)
    xgb.fit(X_train, y_train)
    return xgb


def xgb_train_grid_search(X_train, y_train):
    param_grid = {
        "n_estimators": [50, 80, 100, 150, 200],
        "max_depth": [4, 6, 8],
        "learning_rate": [0.05, 0.1, 0.3, 0.5, 1],
    }

    grid_search = GridSearchCV(
        estimator=XGBClassifier(n_jobs=-1, eval_metric="logloss", tree_method='gpu_hist', device='cuda'),
        param_grid=param_grid,
        scoring="accuracy",
        cv=3,
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    return grid_search.best_estimator_
    

## Классификация проектов

### Загрузим данные

In [8]:
df = load_labeled_data("../Data/Reestr/Размеченные")
df

100%|██████████| 19/19 [00:00<00:00, 151.48it/s]


,id,project_name,label
0,000352ac-d728-470f-b466-dbaf03df82ad,Строительство автомобильной дороги общего поль...,Автомобильные дороги
1,0004719e-2520-4e50-92ab-b26be1e9ed0e,Капитальный ремонт автомобильной дороги по ул....,Автомобильные дороги
2,000f1480-522e-4e22-94fd-539f1e93a22b,Капитальный ремонт автомобильной дороги общего...,Автомобильные дороги
3,0012f905-46ab-4648-b519-2f1f7e2c2b1a,Капитальный ремонт автомобильной дороги Р-241 ...,Автомобильные дороги
4,001ae5dc-0cb7-40be-b52e-5e33dec04fb7,Реконструкция дорожного покрытия ул. Пограничн...,Автомобильные дороги
...,...,...,...
95,06ee6bfe-d776-474c-8209-c4d666aa947f,"Строительство блока отстойников на УППН ""Сухан...",Технологические установки
96,06f42c68-acde-49a5-9cf5-5cb1e4edd89a,Реконструкция опасного производственного объек...,Технологические установки
97,072313c2-4757-49ab-8e42-0aa2e1a3de12,"Кусты №4Б, №15, №59, №64Б Сыморьяхского местор...",Технологические установки
98,077a2af1-b8af-4ce6-bb0e-e8edde2200f4,Обустройство куста скважин № 407б Тагринского ...,Технологические установки


### Векторизуем различными способами

In [9]:
# Векторизация WordToVek
# word2vec_vectors = vectorize_with_word2vec(df['project_name'], 300)

# word2vec_vectors.wv.most_similar('мост')

In [10]:
# Векторизация с помощью SentenceTransformer с использованием модели 'sberbank-ai/sbert_large_nlu_ru'
sbert_vectors = vectorize_with_sentence_transformer(
    "sberbank-ai/sbert_large_nlu_ru", df["project_name"]
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


### Разделим данные на тестовую и обучающую выборки

In [11]:
X_train, X_test, y_train, y_test = data_train_test_split(sbert_vectors, df["label"])

### Классификация 1. Логистическая регрессия + sbert sentence transformer

In [12]:
log_reg_model = log_reg(X_train, y_train)
y_pred = log_reg_model.predict(X_test)

print("Точность:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Точность: 0.7526315789473684
              precision    recall  f1-score   support

           0       0.81      0.89      0.85        19
           1       0.69      0.61      0.65        18
           2       0.45      0.45      0.45        22
           3       0.91      0.83      0.87        24
           4       0.83      0.65      0.73        23
           5       0.65      0.68      0.67        25
           6       1.00      0.88      0.94        17
           7       0.74      0.88      0.80        16
           8       0.56      0.82      0.67        11
           9       0.65      0.87      0.74        23
          10       0.69      1.00      0.81        11
          11       0.76      0.80      0.78        20
          12       0.90      0.95      0.93        20
          13       0.94      0.77      0.85        22
          14       1.00      0.84      0.91        25
          15       0.82      0.78      0.80        18
          16       0.86      0.23      0.36        2

### Вариант классификации 2. XGBoost + sbert sentence transformer

In [13]:
xgb_model = xgb(X_train, y_train)
y_pred = xgb_model.predict(X_test)

print("Точность:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

/usr/local/lib/python3.10/dist-packages/xgboost/sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")


Точность: 0.7973684210526316
              precision    recall  f1-score   support

           0       0.71      0.89      0.79        19
           1       0.80      0.67      0.73        18
           2       0.57      0.59      0.58        22
           3       1.00      0.75      0.86        24
           4       0.84      0.70      0.76        23
           5       0.80      0.80      0.80        25
           6       1.00      0.88      0.94        17
           7       0.73      0.69      0.71        16
           8       0.71      0.91      0.80        11
           9       0.68      0.91      0.78        23
          10       0.92      1.00      0.96        11
          11       0.81      0.65      0.72        20
          12       0.83      0.95      0.88        20
          13       0.87      0.91      0.89        22
          14       0.93      1.00      0.96        25
          15       0.68      0.83      0.75        18
          16       0.93      0.50      0.65        2

### Вариант классификации. XGBoost + sbert sentence transformer + оптимизация гипперпараметров с GridSearch

In [ ]:
xgb_model_gs = xgb_train_grid_search(X_train, y_train)
y_pred = xgb_model_gs.predict(X_test)

print("Точность:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))